# **How are CO-OPS Sea Level Trends calculated?**

Last updated on: 12/19/2025

## Overview
NOAA's Center for Operational Oceanographic Products and Services (CO-OPS) is the nation's source for [coastal inundation data and sea level trends](https://tidesandcurrents.noaa.gov/sea_level_info.html) through its network of long-term water level gauges. This notebook will walk through how [sea level trends](https://tidesandcurrents.noaa.gov/sltrends/faq.html#q2) are calculated.

This notebook is designed to quickly calculate a single period of record (POR) trend or a split trend with a breakpoint at 1993. Users can choose their station of interest here: [Sea Level Trends Map](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html).

For this example we will use [Sewells Point, VA (8638610)](https://tidesandcurrents.noaa.gov/sltrends/sltrends_station.shtml?id=8638610).

## Prerequisites and running the Notebook
This notebook uses Python programming and users may run it either in **Google Colab** or **Jupyter Notebook**.

* **Jupyter Notebook:** Recommended if you prefer to work offline and manage libraries and dependencies personally.
* **Google Colab:** The best option for cloud-based instant access, seamless cooperation, and if you have a good internet connection.

This notebook is designed so that people with limited familiarity with Python can run it.

### Using Google Colab

There are two ways to open this notebook in Colab.

**Method 1: Open directly from GitHub (Recommended)**

This is the easiest way to *open* the notebook.

1.  Open [Google Colab](https://colab.research.google.com/).
2.  In the menu bar, go to **File > Open Notebook**.
3.  In the popup window, click on the **GitHub** tab.
4.  Paste the repository URL into the search bar and press Enter:
    `https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks`
5.  Click on `SL_Trend_Calculation_Notebook/SL_Trend_Calculation_Notebook.ipynb` to open it.

**Method 2: Download and Upload (All Files)**

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer. This folder contains the notebook (`.ipynb`).
4.  Open [Google Colab](https://colab.research.google.com/).
5.  Go to **File > Upload notebook...** and select the `SL_Trend_Calculation_Notebook.ipynb` file from the folder you just extracted.

### Using Jupyter Notebook

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer.
4.  Start Jupyter Notebook on your local machine and navigate to the extracted folder (`.../SL_Trend_Calculation_Notebook/`) to open `SL_Trend_Calculation_Notebook.ipynb`.

---

### ▶️ Running the Notebook

Once you have the notebook open:

1.  [Explore different station/s](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html) and decide which you want to analyze.
2.  In **Step 3** of the notebook, change the **station variable** and **unit of interest**.
3.  In the top menu, click **Runtime** and choose **Run All**.
4.  You will see a green check mark to the left of each cell as it is successfully executed.

Users should be prepared to commit 1-2 hours to familiarize themselves with the notebook, upload data, run code and examine results.

## Learning Outcomes
In this notebook users will be able to:
  1.  Explore historical sea level data
  2.  Learn how to calculate single Period of Record and split Sea Level (SL) trends

## Data Source and Documentations
Input and output documentation can be found in the [CO-OPS Data API](https://api.tidesandcurrents.noaa.gov/api/prod/), [CO-OPS Derived Product API v0.1](https://api.tidesandcurrents.noaa.gov/dpapi/prod/) and the [CO-OPS Metadata API](https://api.tidesandcurrents.noaa.gov/mdapi/prod/).

Standard templates are available within the [CO-OPS API URL Builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html), but this notebook is designed to give you more flexiblity through building our own API template to generate queries from our Data API, MetaData API, and Derived Product API.

All data used for this notebook come from the CO-OPS API.

## Software
This tutorial uses the following Python packages:
- datetime
- requests
- time
- pandas
- numpy
- plotly
- seaborn
- cmocean
- statsmodels
- warnings

## **Step 1**
## Install and Load Packages
Lets check if you have these packages installed. If not, this cell will install them.

In [1]:
import subprocess, sys
packages = ["datetime","requests","time","pandas","numpy","matplotlib","seaborn","cmocean","statsmodels","warnings","plotly"]
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

Now we load our packages and assign them names to use throughout the notebook.

In [2]:
# @title
import datetime as dt
import requests
import time as time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib as mpl
import seaborn as sns
import cmocean
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.dates as mdates
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
import warnings

#this is to suppress user warnings
warnings.filterwarnings("ignore")

## **Step 2**
## Here we choose the station of interest and measurement unit i.e. 'english' (feet) or 'metric' (meter)
Click [here](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html) to see a map of stations to get Station IDs

### The stations with the longest periods of record (greater than 95 years) are:
|Station ID and Name  | Station ID and Name | Station ID and Name|
|:--------------------|:--------------------|:--------------------|
|1612340	Honolulu, HI	| 8574680	Baltimore, MD	| 8771450	Galveston, TX |
|1617760	Hilo, HI	| 8575512	Annapolis, MD	| 9410170	San Diego, CA |
|8410140	Eastport, ME	|	8594900	Washington, DC	| 9410230	La Jolla, CA|
|8418150	Portland, ME	|	8638610	Sewells Point, VA	| 9410660	Los Angeles, CA|
|8443970	Boston, MA	| 8665530	Charleston, SC	| 9414290 San Francisco, CA|
|8452660	Newport, RI	| 8720030	Fernandina Beach, FL | 9439040	Astoria, OR|
|8518750	The Battery, NY	|	8724580	Key West, FL| 9447130	Seattle, WA|
|8534720	Atlantic City, NJ	|	8727520	Cedar Key, FL | 9450460	Ketchikan, AK|
|8557380	Lewes, DE	| 8729840	Pensacola, FL| 9451600	Sitka, AK |

In [40]:
#input station ID below
station = '8638610'

#options are metric or english
units = 'metric'

#trend options are SINGLE or SPLIT
trendType = 'SINGLE'

### Results will be in Meters and Millimeters/Year if 'metric' is chosen
### Results will be in Feet and Inches/Decade if 'english' is chosen

## **Step 3**
## Next, we pull in our data from the CO-OPS API

### We will start with pulling the monthly Mean Sea Level (MSL) data

In [41]:
# @title

product='sealvltrends'
datum = 'msl'

server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/'

myurl = server + product+'.json?station='+station+'&details=monthlymeans&datum='+datum+'&units='+units

# Use "requests" to get the Monthly MSL Data
urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
mydata = content['data']

station_name = content['stationName']

# Make it a Dataframe
mydf = pd.DataFrame(mydata)

last_year = int(mydf['year'].iloc[-1])

#have data end at the end of 2024
mydf = mydf[mydf['year']<2025]

#check out dataframe
mydf

,year,month,meanDate,mslDeseasonalized,msl,trendLine,upperConfidence,lowerConfidence
0,1927,8,08/15/1927,-0.234,-0.200,-0.300,-0.288,-0.312
1,1927,9,09/15/1927,-0.324,-0.215,-0.300,-0.288,-0.312
2,1927,10,10/15/1927,-0.326,-0.233,-0.300,-0.288,-0.312
3,1927,11,11/15/1927,NaN,NaN,-0.299,-0.287,-0.311
4,1927,12,12/15/1927,NaN,NaN,-0.299,-0.287,-0.311
...,...,...,...,...,...,...,...,...
1164,2024,8,08/15/2024,0.158,0.192,0.169,0.180,0.158
1165,2024,9,09/15/2024,0.345,0.454,0.169,0.180,0.158
1166,2024,10,10/15/2024,0.209,0.302,0.169,0.180,0.158
1167,2024,11,11/15/2024,0.205,0.217,0.170,0.181,0.159


### Next, we can grab the published long term period of record (POR) relative sea level (RSL) trend from CO-OPS DPAPI. This trend is a combination of the rising ocean surface and the vertical land motion (VLM) at the site.

In [42]:
# @title CO-OPS Trend
product = "sealvltrends"
affil = "us"
server = "https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/"

myurl = (server + product + ".json?station=" + station + "&affil=" + affil+'&units='+units)

# Use "requests" to get the Monthly MSL Data
urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
mydata = content['SeaLvlTrends']

# Make it a Dataframe
my_trend = pd.DataFrame(mydata)

station_info = my_trend['stationId'].iloc[0]+' '+my_trend['stationName'].iloc[0]
start_trend = pd.to_datetime(my_trend['startDate'].iloc[0])
end_trend = pd.to_datetime(my_trend['endDate'].iloc[0])

trend_status = 'Yes'
#check out dataframe
my_trend


,stationId,stationName,affil,trendUnits,seasonalUnits,seasonalAverage,latitude,longitude,trendType,autoregressive,autoregressiveError,trend,trendError,y2000_offset,startDate,endDate,monthlyMeans,numberEvents,events,seasonalCycleMonth
0,8638610,Sewells Point,US,mm/yr,meters,0.05,36.942778,-76.328611,SINGLE,0.37,0.03,4.84,0.11,0.0,08/15/1927,12/15/2024,https://api.tidesandcurrents.noaa.gov/dpapi/pr...,0,[],"[{'month': 1, 'seasonalVar': -0.0817, 'seasona..."


In [43]:
#Focusing on the MSL
df_msl=mydf.loc[:,['year','month','msl']]
#placeholder to represent each month
df_msl['day'] = '01'
df_msl['MSL'] = pd.to_numeric(df_msl['msl'],downcast='float')
df_msl['year'] = df_msl['year'].astype('int')
df_msl['month'] = df_msl['month'].astype('int')
df_msl['date'] = pd.to_datetime(dict(year=df_msl.year,month=df_msl.month,day=df_msl.day))

if trendType == 'SPLIT':
  if start_trend > pd.to_datetime('1973-01-01'):
    trend_status = 'No'

## **Step 4**

## **Long-term trends in relative water levels**

## Statistical analysis and plotting of long-term single and split trends

#### Walking through the process of calculating the trends as seen in [CO-OPS Sea Level Trends Product](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html)

#### *** The trend calculations in this notebook may vary from the trend calculations on the website listed above.


*   The primary difference is that this notebook is in Python and the trends on the website are calculated using the R programing language. Python and R have different packages/libraries with different capabilities. It is expected to see some variations in using different languages/packages for running the Autoregressive Integrated Moving Average (ARIMA) model. The ARIMA model call and order in this notebook is the same as that run in R to produce the trends on the website, just with different packages/libraries.

### Now we will calculate our trend using Autoregressive Integrated Moving Average (ARIMA) model

This sub-section will walk through how the SL trends are calculated.

What is an ARIMA model?


*   AR - stands for auto-regression which includes equation terms created based on past data points
*   I - stands for integration or differencing which accounts for the overall trend in the data
*   MA - moving average which is the equation terms of error or noise based on past data points

ARIMA models are represented as ARIMA (p,d,q) where p, d, and q are all parameters that the user has to provide

*  p determines the number of autoregressive (AR) terms
*  d determines the order of differencing
*  q determines the number of moving average (MA) terms

For predicting our sea level trend, we are using the simplest ARIMA model (p,d,q) = (1,0,0)

Click [here](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html#) for documentation on the ARIMA model in python.

In [44]:
if trend_status == 'No':
  print('There is not enough data at this station for a split trend calculation')

else:
  #set index to date
  df_msl.set_index(df_msl.date,inplace=True)

  # We use a monthly frequency ('MS' for month-start).
  start_year = 1850
  end_date = df_msl.index.max()
  full_date_range = pd.date_range(start=f'{start_year}-01-01', end=end_date, freq='MS')

  # Reindex the MSL data to the full date range.
  # This creates a complete time series from 1850.
  # Any months without data in the original df_msl will have their 'MSL' value as NaN.
  msl_full_range = df_msl['MSL'].reindex(full_date_range)
  # Create a new DataFrame for the complete trend timeframe
  trend_timeframe = pd.DataFrame(msl_full_range)
  trend_timeframe.index.name = 'date'

  # Calculate the decimal date for the time variable in the model
  # CORRECT: Consistently calculates based on the middle of the month
  trend_timeframe['decimal_date'] = trend_timeframe.index.year + (trend_timeframe.index.month - 1 + 0.5) / 12

  if trendType == 'SINGLE':
    if 'trend0' in my_trend.columns:
      print('This is an earthquake station and the arima model is adjusted to account for the earthquake.')

      cutoff_date = pd.to_datetime(my_trend['endDate0'].iloc[0])

      # Create the DataFrame to store results, ensuring it has the correct index from the start.
      matrix_data = pd.DataFrame(index=trend_timeframe.index)

      # Calculate the base trend for the entire series first.
      full_trend = trend_timeframe['decimal_date'] - 2000

      # Create a boolean mask to identify which rows are before the cutoff date.
      is_before_cutoff = trend_timeframe.index <= cutoff_date

      # 'time0' is the binary indicator for the period before the cutoff.
      matrix_data['time0'] = is_before_cutoff.astype(int)

      # For time1: if the date is before the cutoff, use the trend value; otherwise, use 0.
      matrix_data['time1'] = np.where(is_before_cutoff, full_trend, 0)

      # For time2: if the date is before the cutoff, use 0; otherwise, use the trend value.
      matrix_data['time2'] = np.where(is_before_cutoff, 0, full_trend)

      matrix_data['month']=trend_timeframe.index.month
      matrix_data['msl']=trend_timeframe['MSL']

    else:
      df_msl = df_msl.set_index(df_msl.month)

      # The updated formula to account for the middle of the month
      trend_timeframe['decimal_date'] = trend_timeframe.index.year + (trend_timeframe.index.month - 1 + 0.5) / 12

    #create matrix for ARIMA model, this is the format that the MSL data is in prior to the ARIMA model
      matrix_data = pd.DataFrame()
      matrix_data['time1']=trend_timeframe['decimal_date']-2000
      matrix_data['month']=trend_timeframe.index.month
      matrix_data['msl']=trend_timeframe['MSL']
    matrix_data['mon1']=matrix_data['month'].apply(lambda x: 1 if x == 1 else 0)
    matrix_data['mon2']=matrix_data['month'].apply(lambda x: 1 if x == 2 else 0)
    matrix_data['mon3']=matrix_data['month'].apply(lambda x: 1 if x == 3 else 0)
    matrix_data['mon4']=matrix_data['month'].apply(lambda x: 1 if x == 4 else 0)
    matrix_data['mon5']=matrix_data['month'].apply(lambda x: 1 if x == 5 else 0)
    matrix_data['mon6']=matrix_data['month'].apply(lambda x: 1 if x == 6 else 0)
    matrix_data['mon7']=matrix_data['month'].apply(lambda x: 1 if x == 7 else 0)
    matrix_data['mon8']=matrix_data['month'].apply(lambda x: 1 if x == 8 else 0)
    matrix_data['mon9']=matrix_data['month'].apply(lambda x: 1 if x == 9 else 0)
    matrix_data['mon10']=matrix_data['month'].apply(lambda x: 1 if x == 10 else 0)
    matrix_data['mon11']=matrix_data['month'].apply(lambda x: 1 if x == 11 else 0)
    matrix_data['mon12']=matrix_data['month'].apply(lambda x: 1 if x == 12 else 0)

    matrix_data = matrix_data.drop('month',axis=1)

    msl = pd.DataFrame(matrix_data['msl'])

    matrix_data = matrix_data.drop('msl',axis=1)

    exog=matrix_data

    if units == 'metric':
      if 'trend0' in my_trend.columns:
        # EQ station logic is more complex and having the unit scale
        #at mm vs m results in less accurate result that does not match published trend
        model=ARIMA(endog=msl,exog=exog,order=(1,0,0))
      else:
        #if metric, we want results in mm instead of m
        model=ARIMA(endog=msl*1000,exog=exog,order=(1,0,0))
    else:
    #if english, we want in inches instead of ft
      model=ARIMA(endog=msl*12,exog=exog,order=(1,0,0))
    model_fit = model.fit()

    print(model_fit.summary())
  else:
    #for piecewise we need to adjust our matrix by making a time2 column that starts at the desired breakpoint
    if 'trend0' in my_trend.columns:
      cutoff_date = pd.to_datetime(my_trend['endDate0'].iloc[0])
      if cutoff_date < pd.to_datetime('1993-01-01'):
        trend_timeframe.loc[trend_timeframe.index<cutoff_date,'MSL']=np.nan
    #create matrix for ARIMA model
    matrix_data = pd.DataFrame()

    matrix_data['time1']=trend_timeframe['decimal_date']
    matrix_data['time1']=matrix_data['time1']-1993
    matrix_data['time2']=matrix_data['time1']
    matrix_data.loc[matrix_data['time1']>0,'time1']=0
    matrix_data.loc[matrix_data['time2']<0,'time2']=0
    matrix_data['month']=trend_timeframe.index.month
    matrix_data['msl']=trend_timeframe['MSL']

    matrix_data['mon1']=matrix_data['month'].apply(lambda x: 1 if x == 1 else 0)
    matrix_data['mon2']=matrix_data['month'].apply(lambda x: 1 if x == 2 else 0)
    matrix_data['mon3']=matrix_data['month'].apply(lambda x: 1 if x == 3 else 0)
    matrix_data['mon4']=matrix_data['month'].apply(lambda x: 1 if x == 4 else 0)
    matrix_data['mon5']=matrix_data['month'].apply(lambda x: 1 if x == 5 else 0)
    matrix_data['mon6']=matrix_data['month'].apply(lambda x: 1 if x == 6 else 0)
    matrix_data['mon7']=matrix_data['month'].apply(lambda x: 1 if x == 7 else 0)
    matrix_data['mon8']=matrix_data['month'].apply(lambda x: 1 if x == 8 else 0)
    matrix_data['mon9']=matrix_data['month'].apply(lambda x: 1 if x == 9 else 0)
    matrix_data['mon10']=matrix_data['month'].apply(lambda x: 1 if x == 10 else 0)
    matrix_data['mon11']=matrix_data['month'].apply(lambda x: 1 if x == 11 else 0)
    matrix_data['mon12']=matrix_data['month'].apply(lambda x: 1 if x == 12 else 0)

    matrix_data = matrix_data.drop('month',axis=1)

    msl = pd.DataFrame(matrix_data['msl'])

    matrix_data = matrix_data.drop('msl',axis=1)

    exog=matrix_data

    if units == 'metric':
      #if metric, we want results in mm instead of m
      model=ARIMA(endog=msl*1000,exog=exog,order=(1,0,0))
    else:
      #if english, want in in instead of ft
      model=ARIMA(endog=msl*12,exog=exog,order=(1,0,0))
    model_fit = model.fit()

    print(model_fit.summary())



                               SARIMAX Results                                
Dep. Variable:                    msl   No. Observations:                 2100
Model:                 ARIMA(1, 0, 0)   Log Likelihood               -6501.527
Date:                Fri, 19 Dec 2025   AIC                          13035.054
Time:                        16:22:51   BIC                          13125.449
Sample:                    01-01-1850   HQIC                         13068.163
                         - 12-01-2024                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         45.8006      3.725     12.297      0.000      38.501      53.101
time1          4.8375      0.108     44.787      0.000       4.626       5.049
mon1         -77.7086      5.897    -13.177      0.0

In [45]:
if trend_status == 'No':
  print('There is not enough data at this station for a split trend calculation')
else:
  seasonal_coefs = [model_fit.params[f'mon{i}'] for i in range(1, 13)]
  seasonal_avg = sum(seasonal_coefs) / 12

  if units == 'metric':
    # Create the adjusted MSL column by subtracting the monthly seasonal mean
    zero_mean_seasonality = [(coef - seasonal_avg)/1000 for coef in seasonal_coefs]
  else:
    zero_mean_seasonality = [(coef - seasonal_avg)/12 for coef in seasonal_coefs]

  # Set this new 'date' column as the DataFrame's index
  df_msl.set_index('date', inplace=True)

  # Now your original code will work correctly because the index is a DatetimeIndex
  msl_adjust = df_msl
  msl_adjust['decimal_date'] = msl_adjust.index.year + (msl_adjust.index.month - 1 + 0.5) / 12

    #function to find y-int
  def getYInt(x1, y1,slope):
        y_intercept = slope * (-x1)+y1
        return (y_intercept)

  if trendType == 'SINGLE':
    #get info from model
    coefs=model_fit.params
    se=model_fit.bse
    if 'trend0' in my_trend.columns:
      if units == 'metric':
        trend_text = f"Pre Earthquake/Datum Shift: {np.round(coefs[2]*1000, 2)} +/- {np.round(se[2]*1000 * 1.96, 2)} mm/yr\n" \
                        f"Post Earthquake/Datum Shift: {np.round(coefs[3]*1000, 2)} +/- {np.round(se[3]*1000 * 1.96, 2)} mm/yr"
        time0 = coefs[1]
        slope_m1 = coefs[2]
        error_m1 = (se[2]*1.96)
        slope_m2 = coefs[3]
        error_m2 = (se[3]*1.96)
        constant = coefs[0]
      else:
      # Convert from inches/yr to inches/decade for display
        pre_trend_decade = np.round(coefs[2] * 10, 2)
        pre_error_decade = np.round(se[2] * 1.96 * 10, 2)
        post_trend_decade = np.round(coefs[3] * 10, 2)
        post_error_decade = np.round(se[3] * 1.96 * 10, 2)
        trend_text = f"Pre Earthquake/Datum Shift: {pre_trend_decade} +/- {pre_error_decade} inches/decade\n" \
                        f"Post Earthquake/Datum Shift: {post_trend_decade} +/- {post_error_decade} inches/decade"
        time0 = coefs[1]/12
        slope_m1 = coefs[2] / 12
        error_m1 = (se[2] * 1.96) / 12
        slope_m2 = coefs[3] / 12
        error_m2 = (se[3] * 1.96) / 12
    else:
    #extracting our trend information from model
      if units == 'metric':
        trend_text=str(np.round(coefs[1],2)) +' +/- '+str(np.round(se[1]*1.96,2))+' mm/yr'
        slope_m = coefs[1]/1000
        error_m = (se[1]*1.96)/1000
      else:
        trend_text = f"{np.round(coefs[1] * 10, 2)} +/- {np.round(se[1] * 1.96 * 10, 2)} in/decade"
        slope_m = (coefs[1] / 12)
        error_m = (se[1] * 1.96 / 12)

    print(trend_text)

    # Create a mapping from month (1–12) to seasonal mean
    month_to_seasonal = dict(zip(range(1, 13), zero_mean_seasonality))

    # Create the adjusted MSL column by subtracting the monthly seasonal mean
    msl_adjust['msl_seasonal_adjust'] = msl_adjust['MSL'].subtract(
        msl_adjust.index.month.map(month_to_seasonal)
    )
    if 'trend0' in my_trend.columns:
      #grabbing the months result from arima model
      if units == 'metric':
        constant = coefs[0]
      #calculating the mean seasonal and adding it to the trend line
        seasonal = seasonal_avg+constant
      else:
        constant = coefs[0] / 12
        seasonal = seasonal_avg / 12 + constant

      year_start = msl_adjust['decimal_date'].iloc[0]
      y_2k_offset0 = my_trend['y2000_offset0'].iloc[0]
      y_2k_offset = my_trend['y2000_offset'].iloc[0]
      lin_y_int1 = getYInt(2000,0,slope_m1)

      msl_adjust['linear_trend'] = slope_m1*msl_adjust['decimal_date']+lin_y_int1+time0

      trend_break = msl_adjust.loc[msl_adjust.index==cutoff_date]

      if len(trend_break) == 0:
          trend_break=msl_adjust[msl_adjust.index>cutoff_date].head(1)

      lin_y_int2 = getYInt(2000,0,slope_m2)
      msl_adjust.loc[msl_adjust.index>cutoff_date,'linear_trend']=slope_m2*msl_adjust['decimal_date']+lin_y_int2
      msl_adjust['linear_trend'] = msl_adjust['linear_trend']+seasonal
    else:
      #grabbing the months result from arima model
      seasonal = coefs[2:14]
      if units == 'metric':
        constant = coefs[0]/1000
        #calculating the mean seasonal and adding it to the trend line
        seasonal = np.mean(seasonal)/1000+constant
      else:
        constant = coefs[0] / 12
        seasonal = np.mean(seasonal) / 12 + constant
      lin_y_int = getYInt(2000,seasonal,slope_m)

      #calculating liner trend
      msl_adjust['linear_trend'] = slope_m*msl_adjust['decimal_date']+lin_y_int
    #manually calculate se of estimate
    msl_adjust['y_y2']=(msl_adjust['msl_seasonal_adjust']-msl_adjust['linear_trend'])**2
    y_y2_sum = msl_adjust['y_y2'].sum()
    msl_adjust['se_regression'] = np.sqrt(y_y2_sum / (len(msl_adjust) - 2))

    #calculate standard error of the regression line at each point
    one_over_n = 1/len(msl_adjust)
    msl_adjust['x_x2'] = (msl_adjust['decimal_date']-msl_adjust['decimal_date'].mean())**2
    msl_adjust['sum_x_x2'] = msl_adjust['x_x2'].sum()
    msl_adjust['sqrt'] = np.sqrt(one_over_n+(msl_adjust['x_x2']/msl_adjust['sum_x_x2']))
    msl_adjust['se_point'] = (msl_adjust['se_regression'])*msl_adjust['sqrt']

    # get variance inflation factor (VIF)
    p1 = coefs[-2]
    VIF = ((1 + p1) / (1 - p1)) ** 0.5
    # Calculate standard error of the regression line at each point
    msl_adjust['se_point_vif'] = msl_adjust['se_point'] * VIF

    #confidence bands
    msl_adjust['upr'] = msl_adjust['linear_trend']+1.96*msl_adjust['se_point_vif']
    msl_adjust['lwr'] = msl_adjust['linear_trend']-1.96*msl_adjust['se_point_vif']

  else:
    #for piecewise we need to adjust our matrix by making a time2 column that starts at the desired breakpoint
    if 'trend0' in my_trend.columns:
      if cutoff_date < pd.to_datetime('1993-01-01'):
        trend_timeframe.loc[trend_timeframe.index<cutoff_date,'MSL']=np.nan
    #create matrix for ARIMA model
    matrix_data = pd.DataFrame()

    matrix_data['time1']=trend_timeframe['decimal_date']
    matrix_data['time1']=matrix_data['time1']-1993
    matrix_data['time2']=matrix_data['time1']
    matrix_data.loc[matrix_data['time1']>0,'time1']=0
    matrix_data.loc[matrix_data['time2']<0,'time2']=0
    matrix_data['month']=trend_timeframe.index.month
    matrix_data['msl']=trend_timeframe['MSL']

    matrix_data['mon1']=matrix_data['month'].apply(lambda x: 1 if x == 1 else 0)
    matrix_data['mon2']=matrix_data['month'].apply(lambda x: 1 if x == 2 else 0)
    matrix_data['mon3']=matrix_data['month'].apply(lambda x: 1 if x == 3 else 0)
    matrix_data['mon4']=matrix_data['month'].apply(lambda x: 1 if x == 4 else 0)
    matrix_data['mon5']=matrix_data['month'].apply(lambda x: 1 if x == 5 else 0)
    matrix_data['mon6']=matrix_data['month'].apply(lambda x: 1 if x == 6 else 0)
    matrix_data['mon7']=matrix_data['month'].apply(lambda x: 1 if x == 7 else 0)
    matrix_data['mon8']=matrix_data['month'].apply(lambda x: 1 if x == 8 else 0)
    matrix_data['mon9']=matrix_data['month'].apply(lambda x: 1 if x == 9 else 0)
    matrix_data['mon10']=matrix_data['month'].apply(lambda x: 1 if x == 10 else 0)
    matrix_data['mon11']=matrix_data['month'].apply(lambda x: 1 if x == 11 else 0)
    matrix_data['mon12']=matrix_data['month'].apply(lambda x: 1 if x == 12 else 0)

    matrix_data = matrix_data.drop('month',axis=1)

    msl = pd.DataFrame(matrix_data['msl'])

    matrix_data = matrix_data.drop('msl',axis=1)

    exog=matrix_data

    if units == 'metric':
      #if metric, we want results in mm instead of m
      model=ARIMA(endog=msl*1000,exog=exog,order=(1,0,0))
    else:
      #if english, want in in instead of ft
      model=ARIMA(endog=msl*12,exog=exog,order=(1,0,0))
    model_fit = model.fit()

    print(model_fit.summary())

    trend_break = msl_adjust.loc[msl_adjust.index=='1993-01-15']

    coefs=model_fit.params
    se=model_fit.bse

    if units == 'metric':
      trend_text1=str(np.round(coefs[1],2)) +' +/- '+str(np.round(se[1]*1.96,2))+' mm/yr'
      trend_text2=str(np.round(coefs[2],2)) +' +/- '+str(np.round(se[2]*1.96,2))+' mm/yr'
      slope_m1 = coefs[1]/1000
      error_m1 = (se[1]*1.96)/1000
      slope_m2 = coefs[2]/1000
      error_m2 = (se[2]*1.96)/1000

    else:
      trend_text1=f"{np.round(coefs[1] * 10, 2)} +/- {np.round(se[1] * 1.96 * 10, 2)} in/decade"
      trend_text2=f"{np.round(coefs[2] * 10, 2)} +/- {np.round(se[2] * 1.96 * 10, 2)} in/decade"
      slope_m1 = coefs[1]/12
      error_m1 = coefs[1]/12
      slope_m2 = coefs[2]/12
      error_m2 = (se[2]*1.96)/12

    msl_30yr = msl_adjust.copy()

    if units == 'metric':
    # Create the adjusted MSL column by subtracting the monthly seasonal mean
      zero_mean_seasonality = [(coef - seasonal_avg)/1000 for coef in seasonal_coefs]
    else:
      zero_mean_seasonality = [(coef - seasonal_avg)/12 for coef in seasonal_coefs]
    # Create a mapping from month (1–12) to seasonal mean
    month_to_seasonal = dict(zip(range(1, 13), zero_mean_seasonality))

    # Create the adjusted MSL column by subtracting the monthly seasonal mean
    msl_30yr['msl_seasonal_adjust'] = msl_30yr['MSL'].subtract(
        msl_30yr.index.month.map(month_to_seasonal)
    )

    year_start = msl_30yr['decimal_date'].iloc[0]

    #calculating split trend
    #grabbing the months result from arima model
    seasonal = coefs[3:15]
    if units == 'metric':
      constant = coefs[0]/1000
    #calculating the mean seasonal and adding it to the trend line
      seasonal = np.mean(seasonal)/1000+constant
    else:
      constant = coefs[0] / 12
      seasonal = np.mean(seasonal) / 12 + constant
    year_start = msl_adjust['decimal_date'].iloc[0]

    lin_y_int1 = getYInt(1993,0,slope_m1)

    msl_30yr['piece_trend'] = slope_m1*msl_30yr['decimal_date']+lin_y_int1

    trend_break = msl_30yr.loc[msl_30yr.index=='1993-01-15']

    if len(trend_break) == 0:
        trend_break=msl_30yr[msl_30yr.index>'1993-01-15'].head(1)

    #lin_y_int2 = getYInt(trend_break.index.year,trend_break['piece_trend'].values[0],slope_m2)
    lin_y_int2 = getYInt(1993,0,slope_m2)
    #changing our trend slope at 1993
    msl_30yr.loc[msl_30yr.index>'1992-12-15','piece_trend']=slope_m2*msl_30yr['decimal_date']+lin_y_int2

    msl_30yr['piece_trend']=msl_30yr['piece_trend']+seasonal
    #manually calculate se of estimate
    msl_30yr['y_y2']=(msl_30yr['msl_seasonal_adjust']-msl_30yr['piece_trend'])**2
    y_y2_sum = msl_30yr['y_y2'].sum()
    msl_30yr['se_regression'] = np.sqrt(y_y2_sum / (len(msl_30yr) - 2))

    #calculate standard error of the regression line at each point
    one_over_n = 1/len(msl_30yr)
    msl_30yr['x_x2'] = (msl_30yr['decimal_date']-msl_30yr['decimal_date'].mean())**2
    msl_30yr['sum_x_x2'] = msl_30yr['x_x2'].sum()
    msl_30yr['sqrt'] = np.sqrt(one_over_n+(msl_30yr['x_x2']/msl_30yr['sum_x_x2']))
    msl_30yr['se_point'] = (msl_30yr['se_regression'])*msl_30yr['sqrt']

    # get variance inflation factor (VIF)
    p1 = coefs[-2]
    VIF = ((1 + p1) / (1 - p1)) ** 0.5
    # Calculate standard error of the regression line at each point
    msl_30yr['se_point_vif'] = msl_30yr['se_point'] * VIF

    #confidence bands
    msl_30yr['upr_piece'] = msl_30yr['piece_trend']+1.96*msl_30yr['se_point_vif']
    msl_30yr['lwr_piece'] = msl_30yr['piece_trend']-1.96*msl_30yr['se_point_vif']
    if 'trend0' in my_trend.columns:
      if cutoff_date < pd.to_datetime('1993-01-01'):
        msl_30yr.loc[msl_30yr.index<cutoff_date,'piece_trend']=np.nan
        msl_30yr.loc[msl_30yr.index<cutoff_date,'upr_piece']=np.nan
        msl_30yr.loc[msl_30yr.index<cutoff_date,'lwr_piece']=np.nan


4.84 +/- 0.21 mm/yr


In [46]:
# Create figure
if trendType == 'SINGLE':
  fig = go.Figure()

  # Add traces
  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["msl_seasonal_adjust"],
                          mode='lines', name='Monthly mean sea level with the average seasonal cycle removed',
                          line=dict(color='cornflowerblue')))

  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["linear_trend"],
                          mode='lines', name='Linear Relative Sea Level Trend',
                          line=dict(color='red')))

  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["upr"],
                          mode='lines', name='Upper 95% Confidence Interval',
                          line=dict(color='black')))

  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["lwr"],
                          mode='lines', name='Lower 95% Confidence Interval',
                          line=dict(color='black')))

  # Y-axis label and limits
  if units == 'metric':
      yaxis_title = "Meters"
      if station in ['8771450', '8761724']:
          y_range = [-0.75, 0.6]
      elif station == '9453220':
          y_range = [-0.45, 1.35]
      elif station == '9455090':
          y_range = [-1.35, 0.45]
      else:
          y_range = [-0.6, 0.6]
  else:
      yaxis_title = "Feet"
      y_range = [-2.5, 2] if station in ['8771450','8761724'] else [-2, 2]

  # Update layout with grid lines
  fig.update_layout(
      title=dict(
          text=f"Relative Sea Level Trend<br><br>{station_info} &nbsp;&nbsp;&nbsp;&nbsp; {trend_text}",
          x=0.5,  # Center the title
          xanchor='center',
          font=dict(size=15)
      ),
      xaxis_title="Year",
      yaxis_title=yaxis_title,
      xaxis=dict(showgrid=True),
      yaxis=dict(showgrid=True, range=y_range, zeroline=True, zerolinecolor='black'),
      template="simple_white",
      legend=dict(x=0.01, y=0.99),
      width=1250,
      height=650
  )

  # Show the figure
  fig.show()

else:
  # Ensure datetime index
  msl_30yr.index = pd.to_datetime(msl_30yr.index)

  # Check for trend applicability
  if trend_status == 'No':
    print('No '+trendType+' trends available. Please chose another station or switch the trendType parameter.')
  else:
      fig = go.Figure()


      fig.add_trace(go.Scatter(
          x=msl_30yr.index, y=msl_30yr["msl_seasonal_adjust"],
          name='Monthly mean sea level',
          line=dict(color='cornflowerblue')
      ))
      fig.add_trace(go.Scatter(
          x=msl_30yr.index, y=msl_30yr["piece_trend"],
          name='Trend', line=dict(color='red')
      ))
      fig.add_trace(go.Scatter(
          x=msl_30yr.index, y=msl_30yr["upr_piece"],
          name='Upper CI', line=dict(color='black')
      ))
      fig.add_trace(go.Scatter(
          x=msl_30yr.index, y=msl_30yr["lwr_piece"],
          name='Lower CI', line=dict(color='black')
      ))

      # Add vertical line at 1993 using a transparent scatter trace
      vline_x = pd.Timestamp('1993-01-01')
      y_min = msl_30yr[['msl_seasonal_adjust', 'lwr_piece']].min().min()
      y_max = msl_30yr[['msl_seasonal_adjust', 'upr_piece']].max().max()

      fig.add_trace(go.Scatter(
          x=[vline_x, vline_x],
          y=[y_min, y_max],
          mode="lines",
          line=dict(color="red", dash="dash"),
          name="1993"
      ))

      # Final layout
      fig.update_layout(
          title={
              'text': f"<b>Relative Sea Level Trend</b><br>"
                      f"<span style='font-size:14px'>{station_info}</span><br>"
                      f"<span style='font-size:14px'>Trend before 1993: {trend_text1} "
                      f"&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Trend after 1993: {trend_text2}</span>",
              'x': 0.5,
              'xanchor': 'center',
              'y': 0.95
          },
          xaxis_title="Year",
          #yaxis_title=y_title,
          xaxis=dict(showgrid=True),
          yaxis=dict(showgrid=True),
          legend=dict(x=0.01, y=0.99),
          template='simple_white',
          height=650,
          width=1250
      )

      fig.show()

## **Takeaways**

###Thank you for taking the time to walk through this Sea Level Trend Calculation Notebook

###Some key takeaways include:

*   How to access CO-OPS data from the CO-OPS API (here is a [URL builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html) if you want to explore other CO-OPS data)
*   How to calculate single trends using ARIMA modelling.
*   How to calculate split trends with a breakpoint at 1993 using ARIMA modelling.